In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/tank/klundert/projects/cfdn/prfpy_cfdn/')
import os
import numpy as np
import preprocess
import cortex as cx
import numpy as np
import scipy as sp
import nilearn as nl
from nilearn.surface import load_surf_data
import os, shutil, urllib.request
import cortex as cx
from matplotlib import rc
import nibabel as nb
from nibabel import cifti2
import h5py
import matplotlib.pyplot as plt
from prfpy.stimulus import PRFStimulus2D
from prfpy.model import Iso2DGaussianModel, CSS_Iso2DGaussianModel
from prfpy.fit import Iso2DGaussianFitter, CSS_Iso2DGaussianFitter
from prfpy.utils import Subsurface
from prfpy.stimulus import CFStimulus
from prfpy.model import CFGaussianModel
from prfpy.fit import CFFitter
from prfpy.model import Norm_CFGaussianModel
from prfpy.fit import Norm_CFGaussianFitter
from scipy.optimize import LinearConstraint, NonlinearConstraint
from scipy.io import loadmat
from scipy.ndimage import median_filter, gaussian_filter, binary_propagation
from preprocess import get_cortex
import cfhcpy
from cfhcpy.base import AnalysisBase
from cfhcpy.base import AnalysisBase

/tank/klundert/.local/lib/python3.7/site-packages/nilearn/datasets/__init__.py:96: FutureWarning: Fetchers from the nilearn.datasets module will be updated in version 0.9 to return python strings instead of bytes and Pandas dataframes instead of Numpy arrays.
  "Numpy arrays.", FutureWarning)
/tank/klundert/.local/lib/python3.7/site-packages/nilearn/glm/__init__.py:56: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  'It may change in any future release of Nilearn.', FutureWarning)
/tank/klundert/anaconda3/lib/python3.7/site-packages/tqdm/autonotebook/__init__.py:14: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  " (e.g. in jupyter console)", TqdmExperimentalWarning)


In [3]:
ab = AnalysisBase()
ab.startup(subject='999999', experiment_id="ret", yaml_file="/tank/klundert/projects/hcp_movie/config.yml")
ab.full_data_subjects[0]

Starting analysis of subject 999999 on romulus with settings 
{
 "identifier": "node230",
 "base_dir": "/scratch/2019/visual/hcp_{experiment}/",
 "code_dir": "/tank/tkn219/projects/hcp_movie/",
 "threads": 40
}


100610

In [8]:
subsurface_verts = np.load(f'/scratch/2021/nprf_ss/derivatives/cf-fits/Surface_dm/subsurface_verts.npy')
distance_matrix = np.load(f'/scratch/2021/nprf_ss/derivatives/cf-fits/Surface_dm/distance_matrix.npy')
logvisual_distance_matrix = np.load(f'/scratch/2021/nprf_ss/derivatives/cf-fits/Surface_dm/logvisual_distance_matrix.npy')
visual_distance_matrix = np.load(f'/scratch/2021/nprf_ss/derivatives/cf-fits/Surface_dm/visual_distance_matrix.npy')

In [10]:
sub = f'{ab.full_data_subjects[0]}'
ac = AnalysisBase()
ac.startup(subject=sub, experiment_id="ret", yaml_file="/tank/klundert/projects/hcp_movie/config.yml")
mydat_train_stim = get_cortex(ac._read_tc_data(run=0).T)
mydat_test_stim = get_cortex(ac._read_tc_data(run=1).T)

Starting analysis of subject 100610 on romulus with settings 
{
 "identifier": "node230",
 "base_dir": "/scratch/2019/visual/hcp_{experiment}/",
 "code_dir": "/tank/tkn219/projects/hcp_movie/",
 "threads": 40
}
Getting whole-brain data from: /scratch/2019/visual/hcp_ret/subjects/100610/tfMRI_RETBAR1_*_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries_sg_psc.nii
Getting whole-brain data from: /scratch/2019/visual/hcp_ret/subjects/100610/tfMRI_RETBAR2_*_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries_sg_psc.nii


In [11]:
roi_index_dict = {
    # somatosensory:
    'CS1_4': 8, 'CS2_3a': 53, 'CS3_3b': 9, 'CS4_1': 51, 'CS5_2': 52,
    # auditory:
    'A1': 24, 'PBelt': 124, 'MBelt': 173, 'LBelt': 174, '52': 103, 'RI': 104,
    # low-level visual:
    'V1': 1, 'V2': 4, 'V3': 5,
    # mid-level and high-level visual:
    'V3A': 13, 'V3B': 19, 'IPS1': 17, 'LIPv': 48, 'LIPd': 95, 
    'VIP': 49, 'FEF': 10, 'MST': 2, 'MT': 23, 'LO1': 20, 'LO2': 21, 'LO3': 159
    }

atlas_data = np.concatenate([load_surf_data(
        os.path.join('/tank/klundert/content/data/atlas', f'Q1-Q6_RelatedParcellation210.CorticalAreas_dil_Colors.59k_fs_LR.dlabel.{hemi}.gii'))
         for hemi in ['L', 'R']])
atlas_data_both_hemis = np.mod(atlas_data, 180)
LO1mask = atlas_data_both_hemis == roi_index_dict['LO1']

In [ ]:
train_stim=CFStimulus(mydat_train_stim, subsurface_verts, logvisual_distance_matrix)
test_stim=CFStimulus(mydat_test_stim, subsurface_verts, logvisual_distance_matrix)

mydat_train = mydat_train_stim[LO1mask]
mydat_test = mydat_test_stim[LO1mask]

model=CFGaussianModel(train_stim)

# Define sigmas
sigmas=np.array([0.5,1,2,3,4,5,7,10,20,30,40,60,80,110])

# Define the fitter
gf_vis = CFFitter(data=mydat_train,model=model)
gf_vis.n_jobs = 25
# Perform the fitting.
gf_vis.grid_fit(sigmas, verbose=True, n_batches=60)

In [13]:
LO1mask.sum()

207

In [ ]:
subs = [100610,
        125525]

In [16]:
ab.full_data_subjects

[100610,
 125525,
 132118,
 145834,
 155938,
 164131,
 169444,
 177140,
 181232,
 191336,
 197348,
 203418,
 221319,
 263436,
 360030,
 395756,
 463040,
 572045,
 644246,
 725751,
 782561,
 833249,
 898176,
 927359,
 995174,
 102311,
 111514,
 126426,
 134627,
 146129,
 156334,
 164636,
 169747,
 177645,
 182436,
 191841,
 198653,
 204521,
 233326,
 283543,
 365343,
 397760,
 467351,
 573249,
 654552,
 732243,
 783462,
 859671,
 899885,
 942658,
 102816,
 114823,
 128935,
 134829,
 146432,
 157336,
 165436,
 171633,
 177746,
 182739,
 192439,
 199655,
 205220,
 239136,
 318637,
 380036,
 401422,
 525541,
 581450,
 671855,
 751550,
 789373,
 861456,
 901139,
 943862,
 104416,
 115017,
 130114,
 135124,
 146735,
 158035,
 167036,
 172130,
 178142,
 185442,
 192641,
 200210,
 209228,
 246133,
 320826,
 381038,
 406836,
 541943,
 601127,
 680957,
 757764,
 814649,
 871762,
 901442,
 105923,
 115825,
 130518,
 137128,
 146937,
 158136,
 167440,
 173334,
 178243,
 186949,
 193845,
 200311,
 